# 14 — PatchTST + DLinear v2 no pH (EF01): protocolo L=2304 + purge/embargo + 5 fatias × 5 seeds

Quarto notebook do protocolo v2 (após 10/11-baselines e 12-LSTNet). PatchTST + DLinearLite do **04** com **UMA** mudança documentada (detalhes no §7–§8); todo o resto espelha o 04. Janelamento/purge/val **idênticos ao 12** (splitter verbatim do §5 do 12). 2025 intocado (benchmark futuro).

## Diff exato vs 04 (`notebooks/04-patchtst-ph.ipynb`)

| | 04 (v1) | 14 (v2, este notebook) |
|---|---|---|
| Janela | `L=8640 → H=288` (30 d) | **`L=2304 → H=288` (8 d, protocolo v2)** |
| Entrada dos modelos | cauda `LN=2016` da janela | **cauda `LN=2016` da janela `L=2304` (arquiteturas e hiperparâmetros idênticos)** |
| Val | 4 fatias por data de fim, sem purge | **5 fatias do 12 + purge/embargo ±H (idêntico ao 12)** |
| Seeds | 1 (42) | **5 seeds `[42, 7, 123, 2024, 999]` — mesmo treino/early-stopping do 04 por seed × 2 modelos (10 treinos)** |
| Régua v1 (referência) | `lstnet-02 val 0,0373` | **`04: dlinear 0,0394 · patchtst 0,0414 · lstnet-02 0,0373` (contexto); média das 5 seeds do 12 lida de CSV (sem re-treino)** |

## Por que UNIVARIADO (limitação declarada)

PatchTST/DLinear do 04 são **univariados por construção**: RevIN por janela + atenção/lineares operando só sobre a série de valor — não há canal de covariável na arquitetura. As covariáveis solar+Fourier entram no LSTNet-v2 (12) e nas features LGBM-v2; portanto a comparação v1×v2 **embute mudança de protocolo** (L, purge, 5ª fatia), não só o modelo. Declarado como limitação.

## Protocolo v2 (travado, idêntico ao 12)

- `L=2304 → H=288` (8 d → 1 d, passo 5 min), interpolação `time` limite 24, descarte de janelas com NaN.
- Val = 5 fatias por data de fim: 19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]** → val 12.960 ([2880, 2880, 2880, 1440, 2880]); treino pós-purge ~59.349 (igual ao 12, mesma variável).
- Purge/embargo: treino exclui janelas cujo alvo `[fim−H, fim]` intersecte qualquer fatia estendida `±H` (gap mín +289 passos). Splitter `purge_train`/`signed_gap_steps` **verbatim** do 12 §5 — ver §5 (trava por `assert`).

## Execução remota (UM job por vez — 12c/23 GB estouram com concorrência)

- **Solo (máquina livre):** `.venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/14-v2-patchtst-ph.ipynb`
- **Compartilhada:** `OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/14-v2-patchtst-ph.ipynb`
- Nunca `sleep` dentro do comando remoto (o canal MCP expira); `pkill -f` com o truque `[.]` (ex.: `pkill -f 'nbconvert.*14-v2-patchtst-p[h]'`).
- Tempo estimado: **10× o treino do 04** — o 04 levou ~4 min solo (patchtst 230 s + dlinear 5 s) → **~50–60 min solo / ~2 h compartilhada** p/ os 10 treinos (base v2/seed ~2,4× maior que a do 04; dlinear é ~50× mais rápido que patchtst).
- Prophet/ARIMA **não** correm aqui (só patchtst + dlinear + 3 baratos de referência).

## Saídas (criadas pela execução)

`resultados/14-v2-patchtst-ph/`: `metricas_val_por_seed.csv` (10 linhas: 2 modelos × 5 seeds, val pooled 12.960) · `metricas_val_media_dp.csv` (média±dp pooled por modelo) · `metricas_val_diaria.csv` (10 linhas: pooled dias-âncora por modelo×seed) · `metricas_por_fatia.csv` (50 linhas: 5 fatias × 2 modelos × 5 seeds) · `metricas_por_fatia_media_dp.csv` (10 linhas: média±dp por fatia×modelo) · `metricas_por_dia.csv` (45 dias-âncora) · `modelos/patchtst_ph_s{seed}.pt` + `modelos/dlinear_ph_s{seed}.pt` (×10) + `modelos/normalizacao.json` · `figs/` 01-eda/02-limpeza/03-stl/04-forecasts/05-mae/06-val-dias/07-curvas-treino (espelho do 04; 04/06 com banda média±dp das 5 seeds por modelo, 07 com 2 painéis × 5 curvas + média±dp). O `README.md` do experimento (formato do 04 + seção “Protocolo v2”) é escrito **após** a execução, com números reais + procedência remota (host + work dir).

Convenção: nada in-place em 00–13 · nada de `src/` · nada de 2025 neste notebook.


In [1]:
import json
import os
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "univariavel" / "dados" / "treino").exists())
CSV = ROOT / "univariavel/dados/treino/ef01-mogi-das-cruzes_ph_2024.csv"
OUT = ROOT / "univariavel" / "resultados" / "14-v2-patchtst-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# Protocolo v2 (travado, idêntico ao 12)
L, H = 2304, 288  # 8 d -> 1 d (passo 5 min)
SEASON = 288
INTERP_LIMIT = 24  # 2 h
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24"),
              ("2024-12-13", "2024-12-22")]
# Arquiteturas do 04 (idênticas; só a janela L muda 8640->2304, cauda LN igual)
LN, HN = 2016, 288
PATCH_P, PATCH_S = 48, 24
D_MODEL, NLAYERS, NHEAD, FF = 64, 3, 4, 128
BATCH, LR = 256, 1e-3
MAX_EP_PT, PAT_PT = 60, 10  # PatchTST: espelho do 04
MAX_EP_DL, PAT_DL = 30, 5   # DLinear: espelho do 04
DL_BATCH = 512  # DLinear: espelho do 04
TRAIN_STRIDE, VAL_STRIDE = 4, 4  # espelho do 04
DROPOUT = 0.1
SEEDS = [42, 7, 123, 2024, 999]  # 42 = padrão do repo; 5 reps p/ média±dp
MODELOS = ["patchtst", "dlinear"]
DEVICE = torch.device("cpu")

print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)
print(f"L={L} H={H} LN={LN} HN={HN} modelos={MODELOS} fatias={len(VAL_SLICES)} seeds={SEEDS}")
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
      "| cpu:", os.cpu_count())
print("OUT:", OUT)


ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu
L=2304 H=288 LN=2016 HN=288 modelos=['patchtst', 'dlinear'] fatias=5 seeds=[42, 7, 123, 2024, 999]
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>'} | cpu: 12
OUT: /home/marcos/temporal-model/resultados/14-v2-patchtst-ph


## 1. Carga


In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()


(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 11665 (11.1%)


,ds,y
count,105121,93456.00000
mean,2024-07-01 12:00:00,5.90729
min,2024-01-01 00:00:00,5.21000
25%,2024-04-01 06:00:00,5.71000
50%,2024-07-01 12:00:00,5.92000
75%,2024-09-30 18:00:00,6.09000
max,2024-12-31 00:00:00,6.68000
std,NaN,0.30356


## 2. EDA


In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("ph EF01 2024 — série completa (treino)")
ax[0].set_ylabel("ph")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")


maior gap: 4971 passos = 414.2 h | gaps > 24 passos: 9


fig salva


## 3. Limpeza


In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots = {len(g)*5/60:.1f} h)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")


slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 6408
  outage 2024-01-16 11:35:00 → 2024-01-18 17:50:00 (652 slots = 54.3 h)
  outage 2024-02-13 17:00:00 → 2024-02-13 17:05:00 (2 slots = 0.2 h)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots = 1.1 h)
  outage 2024-04-29 11:30:00 → 2024-05-02 03:10:00 (765 slots = 63.8 h)
  outage 2024-05-27 12:00:00 → 2024-06-13 16:10:00 (4947 slots = 412.2 h)
  outage 2024-10-19 06:05:00 → 2024-10-19 06:30:00 (6 slots = 0.5 h)
  outage 2024-10-19 13:35:00 → 2024-10-19 13:50:00 (4 slots = 0.3 h)
  outage 2024-11-25 15:15:00 → 2024-11-25 15:20:00 (2 slots = 0.2 h)
  outage 2024-12-03 13:30:00 → 2024-12-03 14:50:00 (17 slots = 1.4 h)
fig salva


## 4. ADF + STL (trecho limpo jul–ago)


In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")


ADF stat=-1.79 p-valor=0.384 → NÃO estacionária


fig salva


## 5. Janelamento (L=2304) + val em 5 fatias + purge/embargo ±H

Idêntico ao 12: janelas por data de **fim**, descarte com NaN pós-interp, treino = janelas válidas fora da val que **sobrevivem ao purge** (alvo `[fim−H, fim]` sem interseção com qualquer fatia estendida `±H`). Funções `purge_train`/`signed_gap_steps` **verbatim** do 12 §5. Trava se o purge falhar (gap < H+1 ou overlap > 0). Esperado pH: treino ~59.349 + val 12.960 ([2880, 2880, 2880, 1440, 2880]) + purge ~3.311.


In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
n_desc_nan = int((~ok).sum())
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date

# --- val por data de fim (5 fatias, incl. dez) ---
is_val = np.zeros(len(ends), dtype=bool)
per_slice = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    per_slice.append(int(m.sum()))
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")


def purge_train(ends, is_val, slices):
    """Descarta treino cujo alvo [fim-H, fim] intersecte fatia estendida ±H.
    Verbatim do 12 §5 (H global)."""
    keep = is_val.copy()
    drop = np.zeros(len(ends), dtype=bool)
    for a, b in slices:
        A = pd.Timestamp(a) - pd.Timedelta(minutes=5 * H)          # ini-H
        B = pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) \
            + pd.Timedelta(minutes=5 * H)                           # fim+H
        tgt0 = ends - pd.Timedelta(minutes=5 * H)
        hit = (~is_val) & (ends >= A) & (tgt0 <= B)  # alvo ∩ [A,B] ≠ ∅
        drop |= hit
    keep[~is_val & ~drop] = True  # treino sobrevivente
    return keep, drop  # keep=True → val ou treino válido


def signed_gap_steps(ends_tr, slices):
    """Distância (passos 5min) do alvo [fim-H,fim] à fatia mais próxima; <0 = overlap.
    Verbatim do 12 §5."""
    if not len(ends_tr):
        return None
    e = ends_tr.values.astype("datetime64[m]").astype(np.int64)  # min
    best = np.full(len(e), 10 ** 12)
    for a, b in slices:
        A = (pd.Timestamp(a).to_datetime64().astype("datetime64[m]").astype(int))
        B = ((pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
             .to_datetime64().astype("datetime64[m]").astype(int))
        s0 = e - H * 5
        ov = (e >= A) & (s0 <= B)
        gap = np.where(e < A, (A - e) // 5, np.where(s0 > B, (s0 - B) // 5, -(np.minimum(e, B) - np.maximum(s0, A)) // 5 - 1))
        best = np.minimum(best, gap)
    return best


keep, drop = purge_train(ends, is_val, VAL_SLICES)
va = np.where(is_val)[0]
tr = np.where(keep & ~is_val)[0]
print(f"treino (pós-purge): {len(tr)} | val: {len(va)} | "
      f"descartadas (NaN): {n_desc_nan} | purge: {int(drop.sum())}")

# --- asserts de cobertura por fatia (pH: fatias limpas → val cheia) ---
esperado = [288 * ((pd.Timestamp(b) - pd.Timestamp(a)).days + 1) for a, b in VAL_SLICES]
assert per_slice == esperado, f"cobertura por fatia fora do esperado: {per_slice} vs {esperado}"
assert len(va) == sum(esperado) == 12960, f"val total inesperada: {len(va)}"
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
assert int(drop.sum()) > 0, "purge removeu zero janelas — lógica inativa?"

# --- trava do purge: gap mín ≥ H+1 e overlap zero ---
gaps = signed_gap_steps(ends[tr], VAL_SLICES)
print(f"gap mín alvo-treino→val: +{int(gaps.min())} passos (exigido ≥ {H + 1}); "
      f"treino c/ alvo∩val: {int((gaps < 0).sum())}")
assert int((gaps < 0).sum()) == 0, "LEAKAGE: há alvo de treino dentro da val!"
assert int(gaps.min()) >= H + 1, f"purge/embargo falhou: gap {int(gaps.min())} < {H + 1}"

# --- dias-âncora (23:55) na val: 45 = 10+10+10+5+10 ---
daily_mask = (ends.time == pd.Timestamp("23:55").time()) & is_val
daily_idx = np.where(daily_mask)[0]
por_dia_ct = [int(((ends[daily_idx].date >= pd.Timestamp(a).date()) &
                   (ends[daily_idx].date <= pd.Timestamp(b).date())).sum())
              for a, b in VAL_SLICES]
print("dias-âncora na val:", len(daily_idx), "por fatia:", por_dia_ct)
assert len(daily_idx) == 45, f"dias-âncora inesperados: {len(daily_idx)}"
assert por_dia_ct == [10, 10, 10, 5, 10], f"âncoras por fatia: {por_dia_ct}"


fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
fatia 2024-12-13 → 2024-12-22: 2880 janelas válidas
treino (pós-purge): 59349 | val: 12960 | descartadas (NaN): 26910 | purge: 3311
gap mín alvo-treino→val: +289 passos (exigido ≥ 289); treino c/ alvo∩val: 0
dias-âncora na val: 45 por fatia: [10, 10, 10, 5, 10]


## 6. Métricas + baselines de referência


In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
print("treino rolante (réguas v1/04 p/ referência: dlinear 0,0394 · patchtst 0,0414 · lstnet-02 0,0373):")
print(pd.DataFrame({m: metricas(Ytr, p) for m, p in cheap_preds(Xtr).items()}).T.round(4).to_string())
print("val rolante (5 fatias):")
print(pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T.round(4).to_string())


treino rolante (réguas v1/04 p/ referência: dlinear 0,0394 · patchtst 0,0414 · lstnet-02 0,0373):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0580  0.0796  0.9967  0.9960
sazonal_naive_288  0.0551  0.0757  0.9491  0.9479
media_movel_288    0.0533  0.0715  0.9145  0.9135
val rolante (5 fatias):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0599  0.0829  1.0229  1.0221
sazonal_naive_288  0.0406  0.0589  0.6990  0.6982
media_movel_288    0.0497  0.0652  0.8478  0.8476


## 7. Janelas nativas LN=2016 (construção do 04, UNIVARIADA)

Mesma construção `Wln`/`rowln`/`monta` do 04: cada janela `L=2304` contribui com sua **cauda `LN=2016`** + alvo `H=288`. Sem canal de covariável — PatchTST/DLinear são univariados por construção (ver limitação no cabeçalho); RevIN por janela idêntica ao 04.


In [8]:
val5 = s.to_numpy().astype(np.float32)
Wln = sliding_window_view(val5, LN)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1
print(f"Wln {Wln.shape} (esperado (_, {LN}))")
print(f"janelas nativas válidas: {int((rowln >= 0).sum())}/{len(ends)}")
assert Wln.shape[1] == LN, Wln.shape
assert int((rowln >= 0).sum()) == len(ends), "cauda LN fora da grade!"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
json.dump({"mode": "revin-per-window", "univariate": True, "LN": LN, "HN": HN, "L": L, "H": H,
           "channels": ["valor"], "val_slices": VAL_SLICES, "seeds": SEEDS,
           "modelos": MODELOS},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("normalizacao.json salva")


def monta(idxs):
    ii = np.asarray(idxs); r = rowln[ii]
    return Wln[r], Y[ii].astype(np.float32)


Xtr_, Ytr_ = monta(tr[::TRAIN_STRIDE])
Xva_, Yva_ = monta(va[::VAL_STRIDE])
print(f"treino: {Xtr_.shape} (stride {TRAIN_STRIDE}) | val: {Xva_.shape} (stride {VAL_STRIDE})")


Wln (103106, 2016) (esperado (_, 2016))
janelas nativas válidas: 75620/75620
normalizacao.json salva
treino: (14838, 2016) (stride 4) | val: (3240, 2016) (stride 4)


## 8. PatchTST + DLinear — mesmo treino/early-stopping do 04, repetido nas 5 seeds × 2 modelos

Arquiteturas **verbatim do 04**: `PatchTST` (patch 48/stride 24 → `d_model=64` × 3 camadas × 4 heads × FF 128, dropout 0,1, RevIN por janela) e `DLinearLite` (pool k=25 + 2 lineares `LN→HN`, RevIN por janela). Hiperparâmetros espelhados (Adam `LR=1e-3`/MSE; patchtst `BATCH=256 MAX 60/PAT 10`; dlinear `BATCH=512 MAX 30/PAT 5`; strides 4/4); por (modelo, seed): fixa `random`/`numpy`/`torch`, DataLoader com `shuffle=True`, salva o melhor `val` em `modelos/{patchtst,dlinear}_ph_s{seed}.pt`. Réguas v1 de referência: **04 dlinear 0,0394 · patchtst 0,0414**.


In [9]:
class PatchTST(nn.Module):
    def __init__(self):
        super().__init__()
        self.N = (LN - PATCH_P) // PATCH_S + 1
        self.proj = nn.Linear(PATCH_P, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, self.N, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, NHEAD, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, NLAYERS)
        self.drop = nn.Dropout(DROPOUT)
        self.head = nn.Linear(self.N * D_MODEL, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        z = self.proj(xn.unfold(1, PATCH_P, PATCH_S)) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu

class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, HN)
        self.lin_s = nn.Linear(LN, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu


CFGS = {
    "patchtst": {"cls": PatchTST, "max_ep": MAX_EP_PT, "pat": PAT_PT, "batch": BATCH},
    "dlinear": {"cls": DLinearLite, "max_ep": MAX_EP_DL, "pat": PAT_DL, "batch": DL_BATCH},
}
print({m: sum(p.numel() for p in CFGS[m]["cls"]().parameters()) for m in MODELOS},
      "(04 tinha patchtst=1639010 dlinear=1161794)")

hists, bests, epochs_best, tempos = {}, {}, {}, {}
t_all = time.time()
for MODELO in MODELOS:
    cfg = CFGS[MODELO]
    hists[MODELO], bests[MODELO], epochs_best[MODELO], tempos[MODELO] = {}, {}, {}, {}
    for SEED in SEEDS:
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        model = cfg["cls"]().to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.MSELoss()
        Xtr_v_, Ytr__ = monta(tr[::TRAIN_STRIDE])
        Xva_v_, Yva__ = monta(va[::VAL_STRIDE])
        tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr_v_), torch.from_numpy(Ytr__)),
                               batch_size=cfg["batch"], shuffle=True)
        va_loader = DataLoader(TensorDataset(torch.from_numpy(Xva_v_), torch.from_numpy(Yva__)),
                               batch_size=512)
        best, patience, hist = float("inf"), 0, {"train": [], "val": []}
        best_ep = 0
        t0 = time.time()
        for ep in range(1, cfg["max_ep"] + 1):
            model.train()
            tl = 0.0
            for xb, yb in tr_loader:
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()
                tl += float(loss.detach()) * len(xb)
            tl /= len(tr_loader.dataset)
            model.eval()
            vl = 0.0
            with torch.no_grad():
                for xb, yb in va_loader:
                    vl += float(loss_fn(model(xb), yb)) * len(xb)
            vl /= len(va_loader.dataset)
            hist["train"].append(tl); hist["val"].append(vl)
            tag = ""
            if vl < best:
                best, patience, best_ep = vl, 0, ep
                torch.save({"state": model.state_dict(), "seed": SEED,
                            "cfg": {"ln": LN, "hn": HN, "patch": [PATCH_P, PATCH_S],
                                    "d_model": D_MODEL, "layers": NLAYERS, "heads": NHEAD}},
                           OUT / "modelos" / f"{MODELO}_ph_s{SEED}.pt")
                tag = " *"
            else:
                patience += 1
            print(f"[{MODELO} s{SEED}] ep {ep:02d} train={tl:.4f} val={vl:.4f}{tag}", flush=True)
            if patience >= cfg["pat"]:
                print(f"[{MODELO} s{SEED}] early stopping na ep {ep} (best val={best:.4f} ep {best_ep})")
                break
        dt = time.time() - t0
        hists[MODELO][SEED], bests[MODELO][SEED] = hist, best
        epochs_best[MODELO][SEED], tempos[MODELO][SEED] = best_ep, dt
        print(f"[{MODELO} s{SEED}] treino em {dt:.0f}s | melhor val={best:.4f} (ep {best_ep})")
print(f"10 treinos (2 modelos × 5 seeds) em {time.time()-t_all:.0f}s")
print(pd.DataFrame({f"{mo}_s{sd}": {"best_val": bests[mo][sd], "best_ep": epochs_best[mo][sd],
                                    "train_s": round(tempos[mo][sd])}
                    for mo in MODELOS for sd in SEEDS}).T.round(4).to_string())


{'patchtst': 1639010, 'dlinear': 1161794} (04 tinha patchtst=1639010 dlinear=1161794)


[patchtst s42] ep 01 train=0.0061 val=0.0025 *


[patchtst s42] ep 02 train=0.0039 val=0.0025


[patchtst s42] ep 03 train=0.0035 val=0.0024 *


[patchtst s42] ep 04 train=0.0032 val=0.0024


...

[dlinear s999] ep 05 train=0.0030 val=0.0025


[dlinear s999] ep 06 train=0.0029 val=0.0025


[dlinear s999] ep 07 train=0.0029 val=0.0025


[dlinear s999] ep 08 train=0.0029 val=0.0025


[dlinear s999] early stopping na ep 8 (best val=0.0024 ep 3)
[dlinear s999] treino em 8s | melhor val=0.0024 (ep 3)
10 treinos (2 modelos × 5 seeds) em 2747s
                best_val  best_ep  train_s
patchtst_s42      0.0024      3.0    549.0
patchtst_s7       0.0024      3.0    544.0
patchtst_s123     0.0024      2.0    518.0
patchtst_s2024    0.0024      3.0    562.0
patchtst_s999     0.0024      2.0    529.0
dlinear_s42       0.0024      4.0      8.0
dlinear_s7        0.0025      5.0     10.0
dlinear_s123      0.0025      3.0      8.0
dlinear_s2024     0.0025      5.0      9.0
dlinear_s999      0.0024      3.0      8.0


## 9. Inferência + tabelas (por seed, média±dp pooled/por fatia/dias-âncora)

Inferência cheia na val (12.960 origens, sem stride) por (modelo, seed); `P_va` é descartado após as métricas (memória). `metricas_val_por_seed.csv` = pooled por (modelo, seed), `metricas_val_media_dp.csv` = média±dp pooled por modelo, `metricas_val_diaria.csv` = pooled dias-âncora por (modelo, seed), `metricas_por_fatia.csv` = 5 fatias × 2 modelos × 5 seeds, `metricas_por_fatia_media_dp.csv` = média±dp por (fatia, modelo), `metricas_por_dia.csv` = 45 dias-âncora. Réguas v1 de referência nos textos (**04 dlinear 0,0394 · patchtst 0,0414**); média das 5 seeds do 12 entra como referência **contextual via leitura de CSV** (sem re-treino; skip elegante se ausente).


In [10]:
@torch.no_grad()
def prevê(model_, idxs, batch=256):
    model_.eval()
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
        outs.append(model_(xb).numpy())
    return np.concatenate(outs)


modelos = {}
for MODELO in MODELOS:
    for SEED in SEEDS:
        m = CFGS[MODELO]["cls"]().to(DEVICE)
        ckpt = torch.load(OUT / "modelos" / f"{MODELO}_ph_s{SEED}.pt", map_location="cpu", weights_only=False)
        m.load_state_dict(ckpt["state"]); m.eval()
        modelos[(MODELO, SEED)] = m
print("checkpoints carregados:", len(modelos))

# --- referência contextual 12 (leitura de CSV, sem re-treino; skip elegante se ausente) ---
REF12 = ROOT / "univariavel" / "resultados" / "12-v2-lstnet-ph" / "metricas_val_media_dp.csv"
try:
    ref12 = pd.read_csv(REF12, index_col=0)
    print("referência contextual 12 (média 5 seeds, L=2304 c/ covariáveis — sem re-treino):")
    print(ref12.to_string())
    ref12_mae = float(ref12.loc["MAE", "media"])
except FileNotFoundError:
    ref12, ref12_mae = None, None
    print(f"referência 12 ausente ({REF12}) — segue sem régua v2 contextual")

t0 = time.time()
rows_seed, rows_dia_seed, rows_f = [], [], []
P_d = {}  # (modelo, seed) -> (45, 288)
va_ends = ends[va]
Yd = Y[daily_idx]
cp_d = cheap_preds(X[daily_idx])
NUM = ["MAE", "RMSE", "MAPE", "sMAPE"]
for MODELO in MODELOS:
    for SEED in SEEDS:
        Pv = prevê(modelos[(MODELO, SEED)], va)  # (12960, 288)
        mm = metricas(Yva, Pv)
        rows_seed.append({"modelo": MODELO, "seed": SEED, "MAE": mm["MAE"], "RMSE": mm["RMSE"],
                          "MAPE": mm["MAPE"], "sMAPE": mm["sMAPE"],
                          "best_epoch": epochs_best[MODELO][SEED],
                          "train_s": round(tempos[MODELO][SEED]),
                          "best_val_mse": bests[MODELO][SEED]})
        for a, b in VAL_SLICES:  # métricas por fatia a partir do mesmo Pv (sem re-inferência)
            d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
            m_va = (va_ends.date >= d0) & (va_ends.date <= d1)
            ii_loc = np.where(m_va)[0]
            mf = metricas(Yva[ii_loc], Pv[ii_loc])
            rows_f.append({"fatia": f"{a}→{b}", "modelo": MODELO, "seed": SEED, **mf})
        Pd = prevê(modelos[(MODELO, SEED)], daily_idx)  # (45, 288)
        P_d[(MODELO, SEED)] = Pd
        md = metricas(Yd, Pd)
        rows_dia_seed.append({"modelo": MODELO, "seed": SEED, **md})
        del Pv
print(f"inferência val+dias em {time.time()-t0:.0f}s")

# --- pooled por (modelo, seed) ---
tab_seed = pd.DataFrame(rows_seed).set_index(["modelo", "seed"]).round(4)
tab_seed.to_csv(OUT / "metricas_val_por_seed.csv")
assert len(tab_seed) == 10, tab_seed.shape
print("=== val pooled por (modelo, seed) — 12.960 origens ===")
print(tab_seed.to_string())

# --- média±dp pooled por modelo ---
tab_md = pd.DataFrame(index=MODELOS)
for c in NUM:
    tab_md[f"{c}_media"] = tab_seed.reset_index().groupby("modelo")[c].mean()[MODELOS].round(4).values
    tab_md[f"{c}_dp"] = tab_seed.reset_index().groupby("modelo")[c].std(ddof=1)[MODELOS].round(4).values
tab_md.to_csv(OUT / "metricas_val_media_dp.csv")
print("=== val pooled média±dp (5 seeds, por modelo) ===")
print(tab_md.to_string())
print("Réguas v1/04 (L=8640, 4 fatias): dlinear 0,0394 · patchtst 0,0414 — "
      f"v2 aqui: dlinear {tab_md.loc['dlinear', 'MAE_media']:.4f}±{tab_md.loc['dlinear', 'MAE_dp']:.4f} · "
      f"patchtst {tab_md.loc['patchtst', 'MAE_media']:.4f}±{tab_md.loc['patchtst', 'MAE_dp']:.4f}"
      + (f" · ref contextual 12: {ref12_mae:.4f}" if ref12_mae is not None else " (12 ausente)"))

# --- pooled dias-âncora por (modelo, seed): espelho do metricas_val_diaria do 04 ---
tab_dd = pd.DataFrame(rows_dia_seed).set_index(["modelo", "seed"]).round(4)
tab_dd.to_csv(OUT / "metricas_val_diaria.csv")
print("=== val dias-âncora pooled por (modelo, seed) — 45 origens ===")
print(tab_dd.to_string())

# --- por fatia: 5 × 2 × 5 ---
tab_f = pd.DataFrame(rows_f, columns=["fatia", "modelo", "seed"] + NUM).round(4)
tab_f.to_csv(OUT / "metricas_por_fatia.csv", index=False)
assert len(tab_f) == 50 and tab_f["fatia"].nunique() == 5, tab_f.shape
assert (tab_f["fatia"] == "2024-12-13→2024-12-22").any(), "fatia dez ausente!"
# Seleção única por chamada (sem g[c] encadeado — pandas ≥2 aborta esse padrão)
means = tab_f.groupby(["fatia", "modelo"])[NUM].mean(numeric_only=True).round(4)
stds = tab_f.groupby(["fatia", "modelo"])[NUM].std(ddof=1, numeric_only=True).round(4)
tab_fmd = means.astype(str) + "±" + stds.astype(str)
tab_fmd.to_csv(OUT / "metricas_por_fatia_media_dp.csv")
print("=== val por fatia — MAE por (modelo, seed) ===")
print(tab_f.pivot_table(index="fatia", columns=["modelo", "seed"], values="MAE").round(4).to_string())
print("=== val por fatia — média±dp por (fatia, modelo) ===")
print(tab_fmd.to_string())

# --- por dia-âncora (45 dias; por modelo: 5 seeds + média±dp + baratos) ---
datas = [str(ends[i].date()) for i in daily_idx]
fatias_d = []
for dt_ in ends[daily_idx].date:
    for a, b in VAL_SLICES:
        if pd.Timestamp(a).date() <= dt_ <= pd.Timestamp(b).date():
            fatias_d.append(f"{a}→{b}"); break
por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cp_d[m][k:k+1]) for k in range(len(Yd))] for m in cp_d},
    index=datas)
mae_d = {(mo, sd): [mae(Yd[k:k+1], P_d[(mo, sd)][k:k+1]) for k in range(len(Yd))]
         for mo in MODELOS for sd in SEEDS}
for (mo, sd), arr in mae_d.items():
    por_dia[f"{mo}_s{sd}"] = arr
for mo in MODELOS:
    stack = np.stack([np.asarray(mae_d[(mo, sd)]) for sd in SEEDS])
    por_dia[f"{mo}_media"] = stack.mean(axis=0)
    por_dia[f"{mo}_dp"] = stack.std(axis=0, ddof=1)
por_dia.insert(0, "fatia", fatias_d)
por_dia.to_csv(OUT / "metricas_por_dia.csv")
assert len(por_dia) == 45 and (por_dia["fatia"] == "2024-12-13→2024-12-22").sum() == 10
print("=== val dias-âncora (45): média das 5 seeds por modelo vs baratos ===")
print(por_dia[["fatia"] + list(cp_d) + [f"{mo}_media" for mo in MODELOS]
              + [f"{mo}_dp" for mo in MODELOS]].round(4).to_string())
for mo in MODELOS:
    ms = tab_seed.loc[mo, "MAE"]
    print(f"Melhor seed {mo} na val pooled: s{ms.idxmin()} = {ms.min():.4f}")


checkpoints carregados: 10
referência contextual 12 (média 5 seeds, L=2304 c/ covariáveis — sem re-treino):
        media      dp
MAE    0.0365  0.0004
RMSE   0.0507  0.0003
MAPE   0.6268  0.0056
sMAPE  0.6269  0.0056


inferência val+dias em 26s
=== val pooled por (modelo, seed) — 12.960 origens ===
                  MAE    RMSE    MAPE   sMAPE  best_epoch  train_s  best_val_mse
modelo   seed                                                                   
patchtst 42    0.0357  0.0496  0.6127  0.6133           3      549        0.0024
         7     0.0350  0.0489  0.6018  0.6014           3      544        0.0024
         123   0.0351  0.0486  0.6028  0.6029           2      518        0.0024
         2024  0.0349  0.0490  0.6000  0.5997           3      562        0.0024
         999   0.0353  0.0493  0.6053  0.6060           2      529        0.0024
dlinear  42    0.0355  0.0499  0.6103  0.6101           4        8        0.0024
         7     0.0358  0.0502  0.6149  0.6148           5       10        0.0025
         123   0.0355  0.0500  0.6103  0.6102           3        8        0.0025
         2024  0.0355  0.0501  0.6103  0.6101           5        9        0.0025
         999   0.0355  0.04

=== val dias-âncora (45): média das 5 seeds por modelo vs baratos ===
                            fatia  persistencia  sazonal_naive_288  media_movel_288  patchtst_media  dlinear_media  patchtst_dp  dlinear_dp
2024-04-19  2024-04-19→2024-04-28        0.0305             0.0262           0.0277          0.0234         0.0208       0.0020      0.0004
2024-04-20  2024-04-19→2024-04-28        0.0146             0.0623           0.0620          0.0418         0.0465       0.0030      0.0011
2024-04-21  2024-04-19→2024-04-28        0.0260             0.0226           0.0217          0.0251         0.0165       0.0029      0.0005
2024-04-22  2024-04-19→2024-04-28        0.0194             0.0237           0.0258          0.0222         0.0180       0.0040      0.0008
2024-04-23  2024-04-19→2024-04-28        0.0491             0.0327           0.0342          0.0303         0.0280       0.0052      0.0012
2024-04-24  2024-04-19→2024-04-28        0.0333             0.0240           0.0286       

## 10. Figuras (espelho do 04; 04/06 com banda média±dp por modelo, 07 com 2 painéis × 5 curvas + média±dp)


In [11]:
# --- 04-forecasts: 3 origens do treino (real × sazonal × patchtst/dlinear média±dp 5 seeds) ---
ks = [0, len(Xtr) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
cp_tr = cheap_preds(Xtr)
for ax, k in zip(axes, ks):
    tf = pd.date_range(ends[tr[k]] - pd.Timedelta(minutes=5*(H-1)), ends[tr[k]], freq="5min")
    ax.plot(tf, Ytr[k], "k-", lw=1.5, label="real")
    ax.plot(tf, cp_tr["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    for mo, ls in [("patchtst", "-"), ("dlinear", ":")]:
        Pk = np.stack([prevê(modelos[(mo, sd)], np.array([tr[k]]))[0] for sd in SEEDS])
        mu, sd_ = Pk.mean(axis=0), Pk.std(axis=0, ddof=1)
        ax.plot(tf, mu, ls, lw=1.1, label=f"{mo} média 5 seeds")
        ax.fill_between(tf, mu - sd_, mu + sd_, alpha=0.15)
    ax.set_title(f"origem {ends[tr[k]]} (réguas v1/04: dl 0,0394 · pt 0,0414)")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

# --- 05-mae: barras pooled (baratos + 10 seeds + 2 médias) ---
fig, ax = plt.subplots(figsize=(8, 5))
cheap_va = pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T["MAE"]
ms = tab_seed["MAE"]
bar = pd.concat([cheap_va,
                 ms.set_axis([f"{a}_s{b}" for a, b in ms.index]),
                 pd.Series({f"{mo}_media": ms.loc[mo].mean() for mo in MODELOS})]).sort_values()
bar.plot.barh(ax=ax)
ax.set_title("MAE na val pooled (5 fatias, v2) — baratos + PatchTST/DLinear por seed (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

# --- 06-val-dias: MAE por dia-âncora (baratos + médias por modelo c/ banda ±dp) ---
fig, ax = plt.subplots(figsize=(12, 3.5))
pdf = por_dia
for col, ls in [("sazonal_naive_288", "--"), ("patchtst_media", "-"),
                ("dlinear_media", "-."), ("persistencia", ":")]:
    if col in pdf.columns:
        ax.plot(pd.to_datetime(pdf.index), pdf[col], ls, lw=1.1, label=col)
for mo in MODELOS:
    ax.fill_between(pd.to_datetime(pdf.index), pdf[f"{mo}_media"] - pdf[f"{mo}_dp"],
                    pdf[f"{mo}_media"] + pdf[f"{mo}_dp"], alpha=0.15)
ax.set_title("ph — MAE por dia-âncora na val (5 fatias sazonais, v2, incl. dez)")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")

# --- 07-curvas-treino: por modelo, 5 seeds (finas) + média±dp (espessa + banda) ---
fig, axes = plt.subplots(2, 1, figsize=(8, 7))
for ax, mo in zip(axes, MODELOS):
    for sd in SEEDS:
        ax.plot(hists[mo][sd]["val"], lw=0.8, alpha=0.5, label=f"val s{sd}")
    Lmax = max(len(hists[mo][sd]["val"]) for sd in SEEDS)
    arr_tr = np.full((len(SEEDS), Lmax), np.nan)
    arr_va = np.full((len(SEEDS), Lmax), np.nan)
    for j, sd in enumerate(SEEDS):
        arr_tr[j, :len(hists[mo][sd]["train"])] = hists[mo][sd]["train"]
        arr_va[j, :len(hists[mo][sd]["val"])] = hists[mo][sd]["val"]
    ep = np.arange(1, Lmax + 1)
    ax.plot(ep, np.nanmean(arr_tr, axis=0), "k--", lw=1.2, label="treino média")
    ax.plot(ep, np.nanmean(arr_va, axis=0), "r-", lw=1.5, label="val média")
    ax.fill_between(ep, np.nanmean(arr_va, axis=0) - np.nanstd(arr_va, axis=0, ddof=1),
                    np.nanmean(arr_va, axis=0) + np.nanstd(arr_va, axis=0, ddof=1),
                    color="r", alpha=0.2, label="val ±dp")
    ax.set_title(f"{mo} v2 — loss por época (5 seeds, média±dp)")
    ax.set_xlabel("época"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-curvas-treino.png")
print("figs salvas")


figs salvas


## 11. Conclusões (preencher com números reais após a execução — README só na execução)

Réguas v2 acima (`metricas_val_por_seed.csv` = primária por (modelo, seed); `metricas_val_media_dp.csv` = pooled média±dp por modelo; `metricas_por_fatia_media_dp.csv` mostra cada fatia, incl. dez). Réguas v1 de referência: **04 dlinear 0,0394 · patchtst 0,0414 · lstnet-02 0,0373** (L=8640, 4 fatias, sem purge) — a comparação v1×v2 embute mudança de protocolo (L, purge, 5ª fatia), não só o modelo; PatchTST/DLinear seguem **univariados** (limitação declarada no cabeçalho). Referência contextual v2: média das 5 seeds do 12 lida de CSV (sem re-treino). Checkpoints por (modelo, seed) em `modelos/` para o benchmark/NNLS futuro.

### Protocolo v2 (resumo p/ o README do experimento)

- Janelas `L=2304 → H=288` (8 d → 1 d, 5 min), interp `time` limite 24, descarte com NaN; val 5 fatias por data de fim (19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]**); purge/embargo ±H (gap mín +289; trava por `assert`).
- Modelos = PatchTST + DLinearLite do 04, univariados: cauda `LN=2016` da janela `L=2304`, mesmos hiperparâmetros/RevIN/early-stopping do 04 por (modelo, seed).
- Treino = `BATCH 256/512`, `LR=1e-3`, patchtst `MAX 60/PAT 10`, dlinear `MAX 30/PAT 5`, strides 4/4, Adam/MSE; seeds `[42, 7, 123, 2024, 999]`; reporte por seed + média±dp pooled, por fatia e dias-âncora.

### Procedência da execução (preencher no commit da execução)

- Host remoto: `temporal-remote` 192.168.1.6 · work dir: `/home/marcos/temporal-model` · data: `2026-09-17` (checkpoints 00:48–01:32, notebook 01:40 −03:00) · threads: `solo — 12c, threads unset, torch CPU 2.14.0`
- Pós-execução: escrever `resultados/14-v2-patchtst-ph/README.md` (formato do 04 + seção “Protocolo v2”), indexar em `resultados/README.md` + `notebooks/README.md` + README §7 — com números reais. Não commitar `modelos/*.pt` (vão ao Release via `scripts/baixar_modelos.sh`).
